

# Proyecto AEMET — Predicción de temperatura

Sistema que descarga datos meteorológicos de la AEMET (Agencia Estatal de Meteorología), los persiste en una base de datos y los utiliza para predecir la temperatura del día siguiente.

## Qué hace

A partir de unas coordenadas (latitud y longitud), devuelve la temperatura prevista para el día siguiente en las estaciones meteorológicas más cercanas a ese punto.


## Mapa de la rama


proyecto_aemet_api/
├── data/                              # Datos almacenados para local
│   ├── ALL_10_YEARS                   # Descarga inicial. Crudo
│   ├── df_limpio_10_years.pkl         # Datos transformados y filtrados
│   └── LAST_UPDATE.txt                # Actualiza para automatizacion
│
├── modelos/                           # 1 modelo entrenado por estacion
│   └── modelos_LightGBM/              # LightGBM entrenados en local
│
└── zona_de_trabajo/                   # Archivos de esta rama
    ├── bd.ipynb                       # Funciones de carga a POSTGRES
    └── carga_eda_modl.ipynb           # Carga, limpieza y modelado





## Los modelos de predicción

Existe un modelo independiente por estación, entrenado con su propio histórico de datos.

Cada modelo toma como entrada los últimos 20 días de temperatura y humedad. Como la AEMET publica con 5 días de retraso, cuando se pide la temperatura de mañana la ventana disponible acaba en hoy-5: el modelo se entrena precisamente para eso (cada ventana de 20 días aprende a predecir el día que está 6 días después de su final).

Antes de guardar los modelos nuevos, los anteriores se mueven a una carpeta histórica de respaldo (solo se conserva una versión anterior, por si algo sale mal).


### Producción (AWS)

En AWS la API solo sirve predicciones. La ingesta y el entrenamiento se hacen con funciones Lambda independientes (no pasan por la API), y los modelos se guardan en S3:

- **RDS PostgreSQL**: la base de datos, mismo esquema que en local (`docker/postgres/bd_meteo_v2.sql`).
- **S3**: dos buckets, uno para los datos crudos (pickles) y otro para los modelos entrenados (.joblib).
- **Lambdas** (código en `proyecto_aemet_api/scripts/lambdas/`, listas para pegar en la consola de AWS):

| Lambda | Frecuencia | Qué hace |
|---|---|---|
| `lambda_ingesta_aemet` | Diaria (EventBridge) | Descarga mediciones del último día disponible (hoy - 5 días) y guarda el pickle crudo en S3 |
| `lambda_procesamiento_ingesta` | Automática (trigger S3 `raw/`) | Lee el pickle, limpia los datos y los inserta en RDS |
| `lambda_ingesta_estaciones` | Mensual (EventBridge) | Descarga el inventario de estaciones y guarda el pickle en S3 |
| `lambda_procesamiento_estaciones` | Automática (trigger S3 `estaciones/`) | Lee el pickle, convierte las coordenadas y actualiza la tabla de estaciones |
| `lambda_entrenamiento_standalone` | Cada 6 meses (EventBridge) | Lee el histórico de RDS, entrena los modelos y los sube a S3 (con respaldo de la versión anterior en la carpeta histórica) |

El flujo es: EventBridge despierta a la Lambda de descarga, que deja el pickle en S3; S3 dispara automáticamente la Lambda de procesamiento, que escribe en RDS. Las mediciones de estaciones que no existan en la tabla `estaciones` se descartan (la clave foránea lo exige), así que la carga inicial del inventario debe hacerse antes que la primera ingesta de mediciones.


## Probar una predicción

Con todo el proyecto en marcha, se puede enviar una petición HTTP de la siguiente forma:

```bash
curl -X POST http://localhost:****/api/v1/prediccion \
  -H "Content-Type: application/json" \
  -d '{"latitud": 40.4168, "longitud": -3.7038, "k": 3, "max_distancia_km": 50}'
```

Y la respuesta será algo así:

```json
{
  "fecha": "2026-08-24",
  "temperatura_ponderada": 30.8,
  "estaciones": [
    {
      "indicativo": "3195",
      "nombre": "Madrid, Retiro",
      "provincia": "Madrid",
      "latitud": 40.411111,
      "longitud": -3.678056,
      "distancia_km": 2.1,
      "fecha": "2026-08-24",
      "temperatura_prevista": 31.4
    }
  ]
}
```

La fecha predicha es siempre mañana. `temperatura_ponderada` mezcla las estaciones según lo cerca que estén del punto pedido (si alguna está a menos de 0.5 km, se devuelve directamente la de esa estación). Todas las formas de probar los endpoints están en [docs/flujo.md](docs/flujo.md#cómo-probar-los-endpoints).





